# Stock Stars V2
Ranks the combined S&P 500, Nasdaq-100 and Dow 30 universe using only **returns + company strength**.

**No 13F / Whale / institutional ownership logic is used.**


In [ ]:
# 1. Imports and configuration
import io
import numpy as np
import pandas as pd
import requests
import yfinance as yf

RETURNS_WEIGHT = 0.50
STRENGTH_WEIGHT = 0.50

RETURN_COMPONENT_WEIGHTS = {
    'ret_1y': 0.30,
    'ret_3y_ann': 0.25,
    'ret_5y_ann': 0.20,
    'relative_strength_1y': 0.10,
    'sharpe': 0.10,
    'risk_quality': 0.05,
}

STRENGTH_COMPONENT_WEIGHTS = {
    'roic': 0.25,
    'roe': 0.20,
    'earnings_growth': 0.20,
    'revenue_growth': 0.15,
    'profit_margin': 0.10,
    'financial_health': 0.10,
}


In [ ]:
# 2. Build the stock universe from S&P 500, Nasdaq-100 and Dow 30
HEADERS = {'User-Agent':'Mozilla/5.0'}

def sp500_members():
    url='https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables=pd.read_html(io.StringIO(requests.get(url,headers=HEADERS,timeout=30).text))
    df=tables[0][['Symbol']].copy()
    df['Index']='S&P 500'
    return df

def nasdaq100_members():
    url='https://en.wikipedia.org/wiki/Nasdaq-100'
    tables=pd.read_html(io.StringIO(requests.get(url,headers=HEADERS,timeout=30).text))
    for t in tables:
        if 'Ticker' in t.columns:
            df=t[['Ticker']].rename(columns={'Ticker':'Symbol'}).copy()
            df['Index']='Nasdaq-100'
            return df
    raise RuntimeError('Nasdaq-100 table not found')

def dow30_members():
    url='https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average'
    tables=pd.read_html(io.StringIO(requests.get(url,headers=HEADERS,timeout=30).text))
    for t in tables:
        if 'Symbol' in t.columns and len(t) >= 25 and len(t) <= 40:
            df=t[['Symbol']].copy()
            df['Index']='Dow 30'
            return df
    raise RuntimeError('Dow 30 table not found')

parts=[sp500_members(),nasdaq100_members(),dow30_members()]
membership=pd.concat(parts,ignore_index=True)
membership['Symbol']=membership['Symbol'].astype(str).str.upper().str.replace('.','-',regex=False)
provenance=(membership.groupby('Symbol')['Index']
            .agg(lambda x: ', '.join(sorted(set(x))))
            .reset_index().rename(columns={'Symbol':'ticker','Index':'membership'}))
universe=provenance['ticker'].tolist()
print('Unique stocks:',len(universe))
display(provenance.head())


In [ ]:
# 3. Price-return metrics
def annualized_return(series, years):
    s=series.dropna()
    if len(s)<2 or s.iloc[0] <= 0:
        return np.nan
    total=s.iloc[-1]/s.iloc[0]
    return total**(1/years)-1

def price_metrics(ticker):
    hist=yf.Ticker(ticker).history(period='5y',auto_adjust=True)
    if hist.empty or 'Close' not in hist:
        return {'ticker':ticker,'price_data_ok':False}
    c=hist['Close'].dropna()
    r=c.pct_change().dropna()
    if len(c)<200:
        return {'ticker':ticker,'price_data_ok':False}
    one_year=c.tail(min(252,len(c)))
    three_year=c.tail(min(756,len(c)))
    five_year=c
    ret_1y=(one_year.iloc[-1]/one_year.iloc[0]-1) if len(one_year)>1 else np.nan
    ret_3y_ann=annualized_return(three_year,3) if len(three_year)>500 else np.nan
    ret_5y_ann=annualized_return(five_year,5) if len(five_year)>900 else np.nan
    avg_ann=r.mean()*252 if len(r)>20 else np.nan
    vol=r.std()*np.sqrt(252) if len(r)>20 else np.nan
    sharpe=avg_ann/vol if pd.notna(vol) and vol>0 else np.nan
    return {
        'ticker':ticker,'price_data_ok':True,
        'ret_1y':ret_1y,'ret_3y_ann':ret_3y_ann,'ret_5y_ann':ret_5y_ann,
        'avg_ann_return':avg_ann,'volatility':vol,'sharpe':sharpe,
    }

spy_hist=yf.Ticker('SPY').history(period='1y',auto_adjust=True)['Close'].dropna()
spy_1y=(spy_hist.iloc[-1]/spy_hist.iloc[0]-1) if len(spy_hist)>1 else np.nan


In [ ]:
# 4. Company-strength metrics
def safe_value(d,key):
    v=d.get(key,np.nan)
    return np.nan if v is None else v

def company_strength_metrics(ticker):
    stock=yf.Ticker(ticker)
    info=stock.info or {}
    fin=stock.financials
    bs=stock.balance_sheet
    cf=stock.cashflow

    roe=safe_value(info,'returnOnEquity')
    profit_margin=safe_value(info,'profitMargins')
    earnings_growth=safe_value(info,'earningsGrowth')
    revenue_growth=safe_value(info,'revenueGrowth')
    debt_to_equity=safe_value(info,'debtToEquity')
    operating_cf=safe_value(info,'operatingCashflow')
    free_cf=safe_value(info,'freeCashflow')

    roic=np.nan
    try:
        ebit=fin.loc['EBIT'].dropna()
        debt=bs.loc['Long Term Debt'].dropna() if 'Long Term Debt' in bs.index else pd.Series(dtype=float)
        equity=bs.loc['Stockholders Equity'].dropna()
        common=ebit.index.intersection(equity.index)
        if not debt.empty:
            common=common.intersection(debt.index)
        if len(common)>0:
            invested=equity.loc[common] + (debt.loc[common] if not debt.empty else 0)
            nopat=ebit.loc[common]*0.79
            vals=(nopat/invested.replace(0,np.nan)).replace([np.inf,-np.inf],np.nan).dropna()
            if not vals.empty:
                roic=float(vals.mean())
    except Exception:
        pass

    return {
        'ticker':ticker,'roic':roic,'roe':roe,'profit_margin':profit_margin,
        'earnings_growth':earnings_growth,'revenue_growth':revenue_growth,
        'debt_to_equity':debt_to_equity,'operating_cashflow':operating_cf,
        'free_cashflow':free_cf,
    }


In [ ]:
# 5. Collect metrics for the entire universe
price_df=pd.DataFrame([price_metrics(t) for t in universe])
strength_df=pd.DataFrame([company_strength_metrics(t) for t in universe])
df=(provenance.merge(price_df,on='ticker',how='left')
    .merge(strength_df,on='ticker',how='left'))
df['relative_strength_1y']=df['ret_1y']-spy_1y

# Risk quality: lower volatility and beta are better. Beta comes from yfinance info only when available.
beta_map={}
for t in universe:
    try:
        beta_map[t]=(yf.Ticker(t).info or {}).get('beta',np.nan)
    except Exception:
        beta_map[t]=np.nan
df['beta']=df['ticker'].map(beta_map)
display(df.head())


In [ ]:
# 6. Percentile scoring helpers
def pct_score(series,higher_is_better=True):
    x=pd.to_numeric(series,errors='coerce')
    s=x.rank(pct=True,method='average')*100
    return s if higher_is_better else 100-s

# Combine volatility and beta into a risk-quality score
vol_score=pct_score(df['volatility'],higher_is_better=False)
beta_distance=(pd.to_numeric(df['beta'],errors='coerce')-1.0).abs()
beta_score=pct_score(beta_distance,higher_is_better=False)
df['risk_quality']=pd.concat([vol_score,beta_score],axis=1).mean(axis=1,skipna=True)

# Financial health rewards positive cash flow and reasonable leverage without pretending missing data is good.
health_parts=pd.DataFrame({
    'ocf':df['operating_cashflow'].apply(lambda x:100 if pd.notna(x) and x>0 else (0 if pd.notna(x) else np.nan)),
    'fcf':df['free_cashflow'].apply(lambda x:100 if pd.notna(x) and x>0 else (0 if pd.notna(x) else np.nan)),
    'debt':pct_score(df['debt_to_equity'],higher_is_better=False),
})
df['financial_health']=health_parts.mean(axis=1,skipna=True)


In [ ]:
# 7. Returns Score (0-100)
return_scores=pd.DataFrame({
    'ret_1y':pct_score(df['ret_1y']),
    'ret_3y_ann':pct_score(df['ret_3y_ann']),
    'ret_5y_ann':pct_score(df['ret_5y_ann']),
    'relative_strength_1y':pct_score(df['relative_strength_1y']),
    'sharpe':pct_score(df['sharpe']),
    'risk_quality':df['risk_quality'],
})
df['returns_score']=sum(return_scores[k].fillna(50)*w for k,w in RETURN_COMPONENT_WEIGHTS.items())

# 8. Company Strength Score (0-100)
strength_scores=pd.DataFrame({
    'roic':pct_score(df['roic']),
    'roe':pct_score(df['roe']),
    'earnings_growth':pct_score(df['earnings_growth']),
    'revenue_growth':pct_score(df['revenue_growth']),
    'profit_margin':pct_score(df['profit_margin']),
    'financial_health':df['financial_health'],
})
df['strength_score']=sum(strength_scores[k].fillna(50)*w for k,w in STRENGTH_COMPONENT_WEIGHTS.items())

# Final score: equal emphasis on performance and company strength
df['final_score']=RETURNS_WEIGHT*df['returns_score'] + STRENGTH_WEIGHT*df['strength_score']


In [ ]:
# 9. Star assignment based on rank in the full universe
# Higher percentile = stronger overall stock.
df['final_percentile']=df['final_score'].rank(pct=True,method='average')*100

def stars_from_percentile(p):
    if pd.isna(p): return 'Unrated'
    if p>=90: return '★★★★★'
    if p>=75: return '★★★★'
    if p>=50: return '★★★'
    if p>=25: return '★★'
    return '★'

df['stars']=df['final_percentile'].apply(stars_from_percentile)
ranked=df.sort_values(['final_score','returns_score','strength_score'],ascending=False).reset_index(drop=True)
ranked.index += 1

cols=['ticker','membership','stars','final_score','returns_score','strength_score',
      'ret_1y','ret_3y_ann','ret_5y_ann','relative_strength_1y','sharpe',
      'roic','roe','earnings_growth','revenue_growth','profit_margin','financial_health']
display(ranked[cols].round(3))


In [ ]:
# 10. Selection lists and exports
five_star=ranked[ranked['stars'].eq('★★★★★')].copy()
four_or_five=ranked[ranked['stars'].isin(['★★★★★','★★★★'])].copy()

print('5-star stocks:',len(five_star))
print('4- or 5-star stocks:',len(four_or_five))
display(five_star[cols].round(3))

ranked.to_csv('stock_stars_ranked.csv',index_label='rank')
five_star.to_csv('stock_stars_5star.csv',index_label='rank')
four_or_five.to_csv('stock_stars_4plus.csv',index_label='rank')
print('Exports written.')
